# 视频分类 SlowFast

In [3]:
import os
import cv2
import random

import numpy as np
from tqdm import tqdm
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [4]:
from pytorchvideo.models.hub import slowfast_r50
import decord
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")

## 一、处理数据

In [5]:
ucf_root_dir = r"/mnt/Data/daihongliang"
ucf_dataset_name = "shop2-clean"
ucf_processed_name = "shop2-Processed"
ucf_train_test_file_dir = "shopTrainTestlist"

In [6]:
def get_train_test_lists(ucf_root_dir, ucf_dataset_name):
    data_dir = os.path.join(ucf_root_dir, ucf_dataset_name)
    class_list = os.listdir(data_dir)
    data_list = []

    for i in range(len(class_list)):
        class_item_dir = os.path.join(data_dir, class_list[i])
        data_list.extend([os.path.join(class_item_dir, it) for it in os.listdir(class_item_dir)])

    train_list, test_list = train_test_split(data_list, test_size=0.2)

    return train_list, test_list

In [7]:
train_data, test_data = get_train_test_lists(ucf_root_dir, ucf_dataset_name)

In [8]:
train_data[:5]

['/mnt/Data/daihongliang/shop2-clean/tearing/video33.mp4',
 '/mnt/Data/daihongliang/shop2-clean/drinking/10月10日 (1)52363632525-1.mp4',
 '/mnt/Data/daihongliang/shop2-clean/tearing/video12 (2).mp4',
 '/mnt/Data/daihongliang/shop2-clean/eating/video-8.mp4',
 '/mnt/Data/daihongliang/shop2-clean/drinking/video-17.mp4']

In [9]:
test_data[:5]

['/mnt/Data/daihongliang/shop2-clean/eating/video444.mp4',
 '/mnt/Data/daihongliang/shop2-clean/drinking/10月12日425345646-1.mp4',
 '/mnt/Data/daihongliang/shop2-clean/hiding/video14.mp4',
 '/mnt/Data/daihongliang/shop2-clean/drinking/10月12日4234235-1.mp4',
 '/mnt/Data/daihongliang/shop2-clean/tearing/10月12日353535235-1.mp4']

## 二、编写 Torch 的 Dataset

In [ ]:
class UCFDataset(Dataset):
    def __init__(self, ucf_root_dir, ucf_dataset_name, ucf_train_test_file_dir, train=False):
        self.num_frames = 32

        # 读取训练标签
        self.classes = {}
        classes_file = os.path.join(ucf_root_dir, ucf_train_test_file_dir, 'classshop.txt')
        with open(classes_file) as fin:
            for row in fin:
                line_info = row.strip().split(' ')
                self.classes[line_info[1]] = int(line_info[0]) - 1

        # 读取数据
        train_data, test_data = get_train_test_lists(ucf_root_dir, ucf_dataset_name)
        self.data = train_data if train else test_data
        
        # 数据处理
        self.transform = T.Compose([
            # T.ToTensor(),
            T.Resize(256),
            lambda x: x / 255.0,
            T.CenterCrop(224),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        vr = decord.VideoReader(self.data[i], num_threads=4)
        total_frames = len(vr)

        if total_frames <= self.num_frames:
            frames = vr.get_batch(range(len(vr))).asnumpy()
        else:
            # 随机截取连续的帧
            max_start = total_frames - self.num_frames
            start_frame = random.randint(0, max_start)
            
            frame_indices = list(range(start_frame, start_frame + self.num_frames))
            frames = vr.get_batch(frame_indices).asnumpy()
            

        frames = torch.tensor(frames).permute(0, 3, 1, 2).float()
        frames = self.transform(frames)

        return frames, self.classes[self.data[i].split('/')[-2]]

In [11]:
trainset = UCFDataset(ucf_root_dir, ucf_dataset_name, ucf_train_test_file_dir, train=True)
testset = UCFDataset(ucf_root_dir, ucf_dataset_name, ucf_train_test_file_dir, train=False)

In [12]:
d, l = trainset.__getitem__(123)
print(d.shape, l)

torch.Size([32, 3, 224, 224]) 2


## 三、编写 Torch 的 DataLoader

In [13]:
def collate_fn(batch):
    data = []
    label = []
    
    # 筛选总帧数小于64的样本
    for d, l in batch:
        num_frames = d.shape[0]
        if num_frames < 32:
            continue  # 跳过帧数不足的样本
        
        data.append(d)
        label.append(l)
    
    # 处理批量中无有效样本的情况
    if len(data) == 0:
        return torch.tensor([]), torch.tensor([])
    
    slow_tensor = [item.clone()[::4] for item in data]
    fast_tensor = [item.clone() for item in data]
    
    # 组装张量（所有样本采样后帧数相同）
    slow_tensor = torch.stack(slow_tensor, dim=0).type(torch.FloatTensor).transpose(1, 2).contiguous()
    fast_tensor = torch.stack(fast_tensor, dim=0).type(torch.FloatTensor).transpose(1, 2).contiguous()
    label_tensor = torch.tensor(label).type(torch.FloatTensor)
    
    return [slow_tensor, fast_tensor], label_tensor

In [14]:
trainloader = DataLoader(trainset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=4)
testloader = DataLoader(testset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=4)

In [15]:
batch_data, batch_labels = next(iter(trainloader))
print("数据形状:", batch_data[0].shape, batch_data[1].shape)
print("标签形状:", batch_labels.shape)

数据形状: torch.Size([8, 3, 8, 224, 224]) torch.Size([8, 3, 32, 224, 224])
标签形状: torch.Size([8])


## 四、设计网络

In [16]:
class SlowFast(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        # 加载预训练的SlowFast R50模型
        self.model = slowfast_r50(pretrained=pretrained)
        
        # 替换分类头以适应你的类别数
        in_features = self.model.blocks[6].proj.in_features
        self.model.blocks[6].proj = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.model(x)

In [17]:
net = SlowFast(4)
o = net(batch_data)
print(o.shape)

torch.Size([8, 4])


## 五、构建训练代码

In [18]:
seed = 1
torch.manual_seed(seed)  # 为CPU设置随机种子
torch.cuda.manual_seed(seed)  # 为当前GPU设置随机种子
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU，为所有GPU设置随机种子
np.random.seed(seed)  # Numpy module.
random.seed(seed)  # Python random module.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [19]:
save_root = "./checkpoints_shop/SlowFast"
num_classes = 4
learning_rate = 1e-4
train_epochs = 100

In [20]:
# 创建检查目录
os.makedirs(os.path.join(save_root, 'acc'), exist_ok=True)
os.makedirs(os.path.join(save_root, 'loss'), exist_ok=True)

In [21]:
model = SlowFast(num_classes)
if torch.cuda.device_count() > 1:
    print(f"使用 {torch.cuda.device_count()} 个GPU进行训练。")
    model = nn.DataParallel(model)  # 包装模型
model = model.cuda()

使用 4 个GPU进行训练。


In [22]:
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)
criterion = nn.CrossEntropyLoss()

In [23]:
def calculate_metrics(y_true, y_pred):
    """计算各种评估指标"""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    return accuracy, precision, recall, micro_f1, macro_f1

In [25]:
best_accuracy = 0
min_loss = float('inf')

for epoch in range(1, train_epochs + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{train_epochs}")
    print(f"{'='*60}")
    
    # 更新学习率
    # scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"当前学习率: {current_lr}")

    # 训练循环
    model.train()
    train_loss, train_correct, train_num = 0.0, 0, 0
    train_pred, train_label = [] , []
    for _, (x, y) in enumerate(tqdm(trainloader, desc="训练中")):
        if not y.shape[0]:
            continue
        x, y = [x[0].cuda(), x[1].cuda()], y.long().cuda()
        
        # 梯度清零
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(x)
        loss = criterion(outputs, y)
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        # 统计信息
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_correct += (predicted == y).sum().item()
        train_num += y.size(0)
        
        train_pred.extend(predicted.cpu().numpy())
        train_label.extend(y.cpu().numpy())

    train_avg_loss = train_loss / len(trainloader)
    train_accuracy, precision, recall, micro_f1, macro_f1 = calculate_metrics(train_label, train_pred)
    print(f"Loss: {train_avg_loss:.4f}, Accuracy: {train_accuracy:.2%}, Precision: {precision:.4f}, Recall: {recall:.4f}, Micro F1: {micro_f1:.4f}, Macro F1: {macro_f1:.4f}")

    # 测试循环
    model.eval()
    test_loss, test_correct_num, test_num = 0.0, 0, 0
    test_pred, test_label = [], []
    
    with torch.no_grad():
        for _, (x, y) in enumerate(tqdm(testloader, desc="测试中")):
            if not y.shape[0]:
                continue
            x, y = [x[0].cuda(), x[1].cuda()], y.long().cuda()
            
            outputs = model(x)
            loss = criterion(outputs, y)
            
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            test_correct_num += (predicted == y).sum().item()
            test_num += y.size(0)
            
            test_pred.extend(predicted.cpu().numpy())
            test_label.extend(y.cpu().numpy())
    
    test_avg_loss = test_loss / len(testloader)
    test_accuracy, precision, recall, micro_f1, macro_f1 = calculate_metrics(test_label, test_pred)
    print(f"Loss: {test_avg_loss:.4f}, Accuracy: {test_accuracy:.2%}, Precision: {precision:.4f}, Recall: {recall:.4f}, Micro F1: {micro_f1:.4f}, Macro F1: {macro_f1:.4f}")

     # 保存最佳准确率模型
    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        save_path = os.path.join(save_root, 'acc', f'best_acc_{epoch}.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")
    
    # 保存最低损失模型
    if test_avg_loss < min_loss:
        min_loss = test_avg_loss
        save_path = os.path.join(save_root, 'loss', f'best_loss_{epoch}.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")

    
    # 正确率超过99%时结束训练
    if test_accuracy >= 0.99:
        print(f"训练准确率已达到 {test_accuracy:.2%}，提前终止训练")
        save_path = os.path.join(save_root, f'early_stop_epoch_{epoch}.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")
        break



Epoch 1/100
当前学习率: 0.0001


训练中: 100%|██████████| 319/319 [14:13<00:00,  2.67s/it]


Loss: 0.5400, Accuracy: 79.94%, Precision: 0.7996, Recall: 0.7994, Micro F1: 0.7994, Macro F1: 0.8010


测试中: 100%|██████████| 80/80 [03:11<00:00,  2.40s/it]


Loss: 0.6872, Accuracy: 79.46%, Precision: 0.8219, Recall: 0.7946, Micro F1: 0.7946, Macro F1: 0.7716
模型已保存: ./checkpoints_shop/SlowFast/acc/best_acc_1.pth (Epoch 1, Acc: 79.46%, Loss: 0.6872)
模型已保存: ./checkpoints_shop/SlowFast/loss/best_loss_1.pth (Epoch 1, Acc: 79.46%, Loss: 0.6872)

Epoch 2/100
当前学习率: 0.0001


...

Loss: 0.1220, Accuracy: 96.84%, Precision: 0.9684, Recall: 0.9684, Micro F1: 0.9684, Macro F1: 0.9651

Epoch 100/100
当前学习率: 0.0001


训练中: 100%|██████████| 319/319 [12:25<00:00,  2.34s/it]


Loss: 0.0214, Accuracy: 99.37%, Precision: 0.9937, Recall: 0.9937, Micro F1: 0.9937, Macro F1: 0.9930


测试中: 100%|██████████| 80/80 [03:15<00:00,  2.44s/it]

Loss: 0.1247, Accuracy: 96.37%, Precision: 0.9645, Recall: 0.9637, Micro F1: 0.9637, Macro F1: 0.9633


In [26]:
import torch
import numpy as np
import cv2
import os
import argparse
from pathlib import Path
import torch.nn as nn


class VideoPredictor:
    def __init__(self, model_path, num_classes=4, class_names_file=None):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.num_classes = num_classes

        # 加载模型
        self.model = SlowFast(num_classes)

        # 加载训练好的权重
        checkpoint = torch.load(model_path, map_location=self.device)

        # 处理多GPU训练保存的模型
        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
        else:
            state_dict = checkpoint

        # 移除module前缀（如果使用DataParallel训练）
        new_state_dict = {}
        for k, v in state_dict.items():
            if k.startswith('module.'):
                new_state_dict[k[7:]] = v
            else:
                new_state_dict[k] = v

        self.model.load_state_dict(new_state_dict)
        self.model.to(self.device)
        self.model.eval()

        # 加载类别名称
        self.class_names = self.load_class_names(class_names_file)

    def load_class_names(self, class_names_file):
        """加载类别名称"""
        if class_names_file and os.path.exists(class_names_file):
            class_names = {}
            with open(class_names_file, 'r') as f:
                for line in f:
                    idx, name = line.strip().split(' ')
                    class_names[int(idx) - 1] = name
            return class_names
        else:
            # 如果没有类别文件，返回数字索引
            return {i: f"Class_{i}" for i in range(self.num_classes)}

    def preprocess_video(self, video_path, num_frames=16):
        """预处理视频，提取帧并进行预处理"""
        cap = cv2.VideoCapture(video_path)
        frames = []

        # 读取所有帧
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_resized = cv2.resize(frame_rgb, (112, 112))
            frame_normalized = frame_resized.astype(np.float32)
            frame_chw = np.transpose(frame_normalized, (2, 0, 1))
            frames.append(frame_chw)

        cap.release()

        if len(frames) == 0:
            raise ValueError(f"无法从视频 {video_path} 中读取帧")

        # 采样策略：从视频中均匀采样num_frames帧
        total_frames = len(frames)
        if total_frames >= num_frames:
            # 均匀采样
            indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
            sampled_frames = [frames[i] for i in indices]
        else:
            # 如果帧数不足，重复最后一帧
            sampled_frames = frames
            while len(sampled_frames) < num_frames:
                sampled_frames.append(frames[-1])

        # 转换为tensor
        frames_tensor = torch.tensor(np.stack(sampled_frames), dtype=torch.float32)
        frames_tensor = frames_tensor.unsqueeze(0)  # 添加batch维度
        frames_tensor = frames_tensor.transpose(1, 2)  # (B, T, C, H, W) -> (B, C, T, H, W)

        return frames_tensor

    def predict(self, video_path):
        """对单个视频进行预测"""
        try:
            # 预处理视频
            input_tensor = self.preprocess_video(video_path)
            input_tensor = input_tensor.to(self.device)

            # 预测
            with torch.no_grad():
                outputs = self.model(input_tensor)
                probabilities = torch.softmax(outputs, dim=1)
                predicted_class = torch.argmax(outputs, dim=1).item()
                confidence = probabilities[0][predicted_class].item()

            # 获取类别名称
            class_name = self.class_names.get(predicted_class, f"Class_{predicted_class}")

            # 获取top-3预测结果
            top3_probs, top3_indices = torch.topk(probabilities, 3)
            top3_results = []
            for i in range(3):
                idx = top3_indices[0][i].item()
                prob = top3_probs[0][i].item()
                name = self.class_names.get(idx, f"Class_{idx}")
                top3_results.append((name, prob))

            return {
                'predicted_class': predicted_class,
                'class_name': class_name,
                'confidence': confidence,
                'top3_predictions': top3_results,
                'all_probabilities': probabilities.cpu().numpy()[0]
            }

        except Exception as e:
            return {'error': str(e)}

    def predict_batch(self, video_dir):
        """批量预测目录中的所有视频"""
        video_extensions = ['.avi', '.mp4', '.mov', '.mkv']
        video_files = []

        for ext in video_extensions:
            video_files.extend(Path(video_dir).glob(f'*{ext}'))
            video_files.extend(Path(video_dir).glob(f'*{ext.upper()}'))

        results = {}
        for video_file in video_files:
            print(f"处理视频: {video_file.name}")
            result = self.predict(str(video_file))
            results[video_file.name] = result

        return results

    def save_prediction_video(self, video_path, output_path):
        """保存带预测结果的视频"""
        try:
            # 先进行一次预测获取结果
            result = self.predict(video_path)
            if 'error' in result:
                print(f"预测错误: {result['error']}")
                return None

            # 打开视频文件
            cap = cv2.VideoCapture(video_path)

            # 获取视频信息
            fps = cap.get(cv2.CAP_PROP_FPS)
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            # 设置视频写入器
            fourcc = cv2.VideoWriter_fourcc(*'XVID')
            out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

            print(f"视频信息: {width}x{height}, {fps}fps, 总帧数: {total_frames}")
            print(f"预测结果: {result['class_name']} (置信度: {result['confidence']:.4f})")

            frame_count = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                frame_count += 1

                # 在帧上绘制预测结果
                cv2.putText(frame, f"Prediction: {result['class_name']}", (20, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.putText(frame, f"Confidence: {result['confidence']:.4f}", (20, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.putText(frame, f"Frame: {frame_count}/{total_frames}", (20, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

                # 绘制Top-3预测结果
                for i, (name, prob) in enumerate(result['top3_predictions']):
                    y_pos = 120 + i * 30
                    color = (0, 255, 0) if i == 0 else (255, 255, 0)
                    cv2.putText(frame, f"{i + 1}. {name}: {prob:.4f}", (20, y_pos),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1)

                # 写入帧
                out.write(frame)

                # 显示进度
                if frame_count % 50 == 0:
                    print(f"处理进度: {frame_count}/{total_frames} 帧")

            # 释放资源
            cap.release()
            out.release()

            print(f"带预测结果的视频已保存到: {output_path}")
            return result

        except Exception as e:
            print(f"保存预测视频错误: {str(e)}")
            return {'error': str(e)}


def main():
    parser = argparse.ArgumentParser(description='C3D视频分类预测')
    parser.add_argument('--model_path', type=str, required=True, help='训练好的模型路径')
    parser.add_argument('--video_path', type=str, required=True, help='要预测的视频路径或视频目录')
    parser.add_argument('--class_names', type=str, default=None, help='类别名称文件路径')
    parser.add_argument('--num_classes', type=int, default=101, help='类别数量')
    parser.add_argument('--output_video', type=str, default=None, help='输出视频路径')

    args = parser.parse_args()

    # 初始化预测器
    predictor = VideoPredictor(
        model_path=args.model_path,
        num_classes=args.num_classes,
        class_names_file=args.class_names
    )

    # 判断输入是文件还是目录
    if os.path.isfile(args.video_path):
        if args.output_video:
            # 保存带预测结果的视频
            result = predictor.save_prediction_video(
                args.video_path,
                args.output_video
            )
        else:
            # 单个视频预测
            result = predictor.predict(args.video_path)

        if result and 'error' not in result:
            print(f"\n预测结果:")
            print(f"视频文件: {os.path.basename(args.video_path)}")
            print(f"预测类别: {result['class_name']} (ID: {result['predicted_class']})")
            print(f"置信度: {result['confidence']:.4f}")
            print(f"\nTop-3预测:")
            for i, (name, prob) in enumerate(result['top3_predictions']):
                print(f"  {i + 1}. {name}: {prob:.4f}")
        elif result and 'error' in result:
            print(f"预测错误: {result['error']}")

    elif os.path.isdir(args.video_path):
        # 批量预测
        results = predictor.predict_batch(args.video_path)

        print(f"\n批量预测结果:")
        for video_name, result in results.items():
            if 'error' in result:
                print(f"{video_name}: 错误 - {result['error']}")
            else:
                print(f"{video_name}: {result['class_name']} (置信度: {result['confidence']:.4f})")

    else:
        print(f"错误: {args.video_path} 不是有效的文件或目录")


if __name__ == "__main__":
    # 如果没有命令行参数，可以使用以下代码直接测试
    # main()

    # 或者直接使用以下代码进行测试
    model_path = "./checkpoints_shop/SlowFast/acc/best_acc_51.pth"  # 修改为你的模型路径
    video_path = "/mnt/Data/daihongliang/便利店数据集/drinking/10月3日 (1)141556-1.mp4"  # 修改为你的视频路径
    class_names_file = "/mnt/Data/daihongliang/shopTrainTestlist/classshop.txt"  # 修改为你的类别文件路径
    output_video_path = "output_with_predictions.avi"  # 保存结果视频

    if os.path.exists(model_path):
        predictor = VideoPredictor(
            model_path=model_path,
            num_classes=4,
            class_names_file=class_names_file
        )

        # 方法1: 简单预测（只输出文本结果）
        print("=== 简单预测 ===")
        result = predictor.predict(video_path)

        if 'error' in result:
            print(f"预测错误: {result['error']}")
        else:
            print(f"\n预测结果:")
            print(f"视频文件: {os.path.basename(video_path)}")
            print(f"预测类别: {result['class_name']} (ID: {result['predicted_class']})")
            print(f"置信度: {result['confidence']:.4f}")
            print(f"\nTop-3预测:")
            for i, (name, prob) in enumerate(result['top3_predictions']):
                print(f"  {i + 1}. {name}: {prob:.4f}")

        # 方法2: 保存带预测结果的视频
        print("\n=== 保存预测视频 ===")
        result = predictor.save_prediction_video(
            video_path,
            output_video_path
        )

    else:
        print(f"模型文件不存在: {model_path}")

=== 简单预测 ===
预测错误: input for MultiPathWayWithFuse needs to be a list of tensors

=== 保存预测视频 ===
预测错误: input for MultiPathWayWithFuse needs to be a list of tensors
